# Bitbrain Open Access Sleep (BOAS) Dataset - Download and Processing

This notebook downloads and processes the BOAS dataset from OpenNeuro.

**Dataset Information:**
- Dataset ID: ds005555
- Size: 33.45 GB
- 128 recordings from 108 unique participants
- Simultaneous PSG and wearable EEG recordings
- Human consensus and AI sleep stage labels

**Download Link:** https://openneuro.org/datasets/ds005555/versions/1.1.0

## 1. Install Required Packages

In [ ]:
!pip install openneuro-py mne pandas numpy matplotlib seaborn pyEDFlib

## 2. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import mne
from google.colab import drive
import openneuro

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 3. Mount Google Drive (Optional - for saving processed data)

In [ ]:
# Uncomment if you want to save data to Google Drive
# drive.mount('/content/drive')
# SAVE_PATH = '/content/drive/MyDrive/BOAS_Dataset'
# os.makedirs(SAVE_PATH, exist_ok=True)

## 4. Download Dataset from OpenNeuro

**Note:** The full dataset is 33.45 GB. You can:
- Download specific subjects only
- Download the entire dataset (may take time)
- Download to Google Drive if you have space

In [ ]:
# Dataset configuration
DATASET_ID = 'ds005555'
VERSION = 'v1.1.0'
BASE_PATH = '/content/BOAS_Dataset'

# Create directory
os.makedirs(BASE_PATH, exist_ok=True)

print(f"Downloading dataset {DATASET_ID} to {BASE_PATH}...")
print("This may take a while depending on your selection...")

### Option A: Download Specific Subjects (Recommended for testing)

In [ ]:
# Download first 5 subjects for testing
subjects = ['sub-1', 'sub-2', 'sub-3', 'sub-4', 'sub-5']

for subject in subjects:
    print(f"Downloading {subject}...")
    openneuro.download(
        dataset=DATASET_ID,
        target_dir=BASE_PATH,
        include=[f'{subject}/*']
    )

# Download metadata files
print("Downloading metadata files...")
openneuro.download(
    dataset=DATASET_ID,
    target_dir=BASE_PATH,
    include=['participants.tsv', 'participants.json', 'channels.tsv', 
             'channels.json', 'dataset_description.json', 'README']
)

print("Download complete!")

### Option B: Download Entire Dataset (33.45 GB)

In [ ]:
# Uncomment to download the entire dataset
# WARNING: This is 33.45 GB and will take considerable time
# openneuro.download(
#     dataset=DATASET_ID,
#     target_dir=BASE_PATH
# )
# print("Full dataset download complete!")

### Alternative: Direct Download via wget

In [ ]:
# Alternative download method using AWS S3 (OpenNeuro mirror)
# Download specific subject example

# !aws s3 sync --no-sign-request \
#     s3://openneuro.org/ds005555 \
#     {BASE_PATH} \
#     --exclude "*" \
#     --include "sub-1/*" \
#     --include "participants.tsv" \
#     --include "channels.tsv"

## 5. Load and Explore Metadata

In [ ]:
# Load participants metadata
participants_file = os.path.join(BASE_PATH, 'participants.tsv')
if os.path.exists(participants_file):
    participants_df = pd.read_csv(participants_file, sep='\t')
    print("Participants Metadata:")
    print(participants_df.head(10))
    print(f"\nTotal recordings: {len(participants_df)}")
    print(f"Unique participants: {participants_df['pid'].nunique()}")
else:
    print("Participants file not found. Please download it first.")

In [ ]:
# Demographic summary
if 'participants_df' in locals():
    print("\nDemographic Summary:")
    print(f"Age range: {participants_df['age'].min():.0f} - {participants_df['age'].max():.0f} years")
    print(f"Mean age: {participants_df['age'].mean():.1f} ± {participants_df['age'].std():.1f} years")
    print(f"\nSex distribution:")
    print(participants_df['sex'].value_counts())
    print(f"\nBMI range: {participants_df['bmi'].min():.1f} - {participants_df['bmi'].max():.1f}")
    print(f"Mean BMI: {participants_df['bmi'].mean():.1f} ± {participants_df['bmi'].std():.1f}")

## 6. Visualize Demographics

In [ ]:
if 'participants_df' in locals():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Age distribution
    axes[0].hist(participants_df['age'], bins=20, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Age (years)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Age Distribution')
    
    # Sex distribution
    sex_counts = participants_df['sex'].value_counts()
    axes[1].bar(sex_counts.index, sex_counts.values, alpha=0.7)
    axes[1].set_xlabel('Sex')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Sex Distribution')
    
    # BMI distribution
    axes[2].hist(participants_df['bmi'], bins=20, edgecolor='black', alpha=0.7)
    axes[2].set_xlabel('BMI')
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('BMI Distribution')
    
    plt.tight_layout()
    plt.show()

## 7. Load and Process EEG Data

In [ ]:
def load_subject_data(subject_id, data_path=BASE_PATH):
    """
    Load PSG and headband EEG data for a specific subject
    
    Parameters:
    -----------
    subject_id : str
        Subject ID (e.g., 'sub-1')
    data_path : str
        Base path to dataset
    
    Returns:
    --------
    dict with 'psg', 'headband', 'psg_events', 'headband_events'
    """
    subject_path = os.path.join(data_path, subject_id)
    
    # Find EDF files
    psg_file = None
    headband_file = None
    
    for file in os.listdir(subject_path):
        if file.endswith('acq-psg_eeg.edf'):
            psg_file = os.path.join(subject_path, file)
        elif file.endswith('acq-headband_eeg.edf'):
            headband_file = os.path.join(subject_path, file)
    
    data = {}
    
    # Load PSG data
    if psg_file:
        print(f"Loading PSG data from {os.path.basename(psg_file)}...")
        data['psg'] = mne.io.read_raw_edf(psg_file, preload=False)
        
        # Load PSG events
        psg_events_file = psg_file.replace('_eeg.edf', '_events.tsv')
        if os.path.exists(psg_events_file):
            data['psg_events'] = pd.read_csv(psg_events_file, sep='\t')
    
    # Load headband data
    if headband_file:
        print(f"Loading headband data from {os.path.basename(headband_file)}...")
        data['headband'] = mne.io.read_raw_edf(headband_file, preload=False)
        
        # Load headband events
        headband_events_file = headband_file.replace('_eeg.edf', '_events.tsv')
        if os.path.exists(headband_events_file):
            data['headband_events'] = pd.read_csv(headband_events_file, sep='\t')
    
    return data

In [ ]:
# Load first subject as example
subject_data = load_subject_data('sub-1')

if 'psg' in subject_data:
    print("\nPSG Recording Info:")
    print(subject_data['psg'].info)
    print(f"\nChannel names: {subject_data['psg'].ch_names}")
    print(f"Sampling rate: {subject_data['psg'].info['sfreq']} Hz")
    print(f"Duration: {subject_data['psg'].times[-1]/3600:.2f} hours")

## 8. Visualize Sleep Stages

In [ ]:
def plot_hypnogram(events_df, title="Hypnogram"):
    """
    Plot sleep stages over time (hypnogram)
    
    Sleep stages:
    0: Wake
    1: N1
    2: N2
    3: N3
    4: REM
    8: PSG disconnection
    -2: Artifacts/missing
    """
    if 'stage_hum' in events_df.columns:
        stage_col = 'stage_hum'
    elif 'stage_ai' in events_df.columns:
        stage_col = 'stage_ai'
    else:
        print("No sleep stage data found")
        return
    
    plt.figure(figsize=(14, 4))
    
    # Convert onset to hours
    time_hours = events_df['onset'] / 3600
    stages = events_df[stage_col]
    
    # Plot as step plot
    plt.step(time_hours, stages, where='post', linewidth=1.5)
    
    plt.xlabel('Time (hours)')
    plt.ylabel('Sleep Stage')
    plt.title(title)
    plt.yticks([0, 1, 2, 3, 4], ['Wake', 'N1', 'N2', 'N3', 'REM'])
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\nSleep Stage Summary:")
    stage_counts = events_df[stage_col].value_counts().sort_index()
    stage_names = {0: 'Wake', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'REM', 8: 'Disconnect', -2: 'Artifact'}
    
    for stage, count in stage_counts.items():
        duration_min = count * 30 / 60  # 30-second epochs
        print(f"{stage_names.get(stage, f'Stage {stage}')}: {count} epochs ({duration_min:.1f} min)")

In [ ]:
# Plot hypnogram if events data is available
if 'psg_events' in subject_data:
    plot_hypnogram(subject_data['psg_events'], title="PSG Human-Scored Hypnogram - Subject 1")

## 9. Plot EEG Signal

In [ ]:
def plot_eeg_segment(raw, start_time=0, duration=30, channels=None):
    """
    Plot a segment of EEG data
    
    Parameters:
    -----------
    raw : mne.io.Raw
        Raw EEG data
    start_time : float
        Start time in seconds
    duration : float
        Duration to plot in seconds
    channels : list
        List of channel names to plot (None = all)
    """
    if channels is None:
        # Select only EEG channels
        eeg_channels = [ch for ch in raw.ch_names if 'EEG' in ch or 'HB_' in ch]
        channels = eeg_channels[:6]  # Limit to first 6 channels
    
    raw.plot(start=start_time, duration=duration, n_channels=len(channels), 
             scalings='auto', title=f'EEG Signal ({start_time}s - {start_time+duration}s)')
    plt.show()

In [ ]:
# Plot a 30-second segment of PSG data
if 'psg' in subject_data:
    # Select specific EEG channels
    eeg_channels = [ch for ch in subject_data['psg'].ch_names if 'PSG_' in ch and 'EEG' not in ch]
    if len(eeg_channels) > 0:
        plot_eeg_segment(subject_data['psg'], start_time=3600, duration=30, 
                        channels=eeg_channels[:4])

## 10. Compare Human vs AI Scoring

In [ ]:
def compare_scoring(events_df):
    """
    Compare human consensus and AI sleep stage scoring
    """
    if 'stage_hum' not in events_df.columns or 'stage_ai' not in events_df.columns:
        print("Missing scoring data")
        return
    
    # Filter out special codes
    valid_data = events_df[(events_df['stage_hum'] >= 0) & 
                           (events_df['stage_hum'] <= 4) &
                           (events_df['stage_ai'] >= 0) & 
                           (events_df['stage_ai'] <= 4)]
    
    # Calculate agreement
    agreement = (valid_data['stage_hum'] == valid_data['stage_ai']).mean()
    print(f"Human-AI Agreement: {agreement*100:.2f}%")
    
    # Confusion matrix
    from sklearn.metrics import confusion_matrix, classification_report
    
    cm = confusion_matrix(valid_data['stage_hum'], valid_data['stage_ai'])
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Wake', 'N1', 'N2', 'N3', 'REM'],
                yticklabels=['Wake', 'N1', 'N2', 'N3', 'REM'])
    plt.xlabel('AI Predicted')
    plt.ylabel('Human Consensus')
    plt.title('Confusion Matrix: Human vs AI Sleep Scoring')
    plt.tight_layout()
    plt.show()
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(valid_data['stage_hum'], valid_data['stage_ai'],
                                target_names=['Wake', 'N1', 'N2', 'N3', 'REM']))

In [ ]:
# Compare human and AI scoring
if 'psg_events' in subject_data:
    compare_scoring(subject_data['psg_events'])

## 11. Batch Processing Function

In [ ]:
def process_all_subjects(data_path=BASE_PATH, save_path=None):
    """
    Process all subjects in the dataset
    
    Parameters:
    -----------
    data_path : str
        Path to dataset
    save_path : str
        Path to save processed results (optional)
    """
    # Find all subject directories
    subjects = sorted([d for d in os.listdir(data_path) if d.startswith('sub-')])
    
    results = []
    
    for subject in subjects:
        print(f"\nProcessing {subject}...")
        try:
            data = load_subject_data(subject, data_path)
            
            result = {'subject': subject}
            
            # Extract basic info
            if 'psg' in data:
                result['psg_duration_hours'] = data['psg'].times[-1] / 3600
                result['psg_channels'] = len(data['psg'].ch_names)
            
            if 'psg_events' in data:
                events = data['psg_events']
                if 'stage_hum' in events.columns:
                    stage_counts = events['stage_hum'].value_counts()
                    result['wake_epochs'] = stage_counts.get(0, 0)
                    result['n1_epochs'] = stage_counts.get(1, 0)
                    result['n2_epochs'] = stage_counts.get(2, 0)
                    result['n3_epochs'] = stage_counts.get(3, 0)
                    result['rem_epochs'] = stage_counts.get(4, 0)
            
            results.append(result)
            
        except Exception as e:
            print(f"Error processing {subject}: {str(e)}")
    
    # Create summary dataframe
    summary_df = pd.DataFrame(results)
    
    if save_path:
        summary_df.to_csv(os.path.join(save_path, 'processing_summary.csv'), index=False)
    
    return summary_df

In [ ]:
# Process all downloaded subjects
# summary = process_all_subjects(BASE_PATH)
# print(summary)

## 12. Export Processed Data

In [ ]:
def export_subject_features(subject_id, output_path, data_path=BASE_PATH):
    """
    Export processed features for a subject to CSV/NPY format
    """
    data = load_subject_data(subject_id, data_path)
    
    os.makedirs(output_path, exist_ok=True)
    
    # Export sleep stages
    if 'psg_events' in data:
        events_file = os.path.join(output_path, f'{subject_id}_sleep_stages.csv')
        data['psg_events'].to_csv(events_file, index=False)
        print(f"Saved sleep stages to {events_file}")
    
    # Export EEG data (optional - can be large)
    # if 'psg' in data:
    #     eeg_data, times = data['psg'][:, :]
    #     np.save(os.path.join(output_path, f'{subject_id}_psg_data.npy'), eeg_data)
    #     print(f"Saved PSG data to {output_path}")
    
    return output_path

## Next Steps

You can now:
1. Download more subjects as needed
2. Implement custom feature extraction
3. Train machine learning models
4. Analyze sleep architecture
5. Compare PSG vs wearable EEG performance

For questions about the dataset, contact:
- Eduardo López-Larraz: eduardo.lopez@bitbrain.com
- Jens G. Klinzing: jens.klinzing@bitbrain.com